# MemoRAG with Ollama + Ollama Cloud

This notebook implements a MemoRAG-style pipeline with:
- Global memory formation from long context
- Cue-driven retrieval
- Final answer generation

Design goals:
- Use local Ollama and Ollama Cloud LLMs
- Keep functions minimal (mostly linear sections)
- Split code into clear section blocks

In [ ]:
# Section 2: Imports
import os
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama, OllamaEmbeddings

In [ ]:
!ollama ls

In [ ]:
# Section 3: Environment and model configuration
load_dotenv()

# Local Ollama settings
LOCAL_OLLAMA_BASE_URL = os.getenv("LOCAL_OLLAMA_BASE_URL", "http://localhost:11434")
LOCAL_LLM_MODEL = os.getenv("LOCAL_LLM_MODEL", "gemini-3-flash-preview:cloud")
LOCAL_EMBED_MODEL = os.getenv("LOCAL_EMBED_MODEL", "nomic-embed-text")


# Select answer model preference: "cloud" or "local"
ANSWER_MODEL_SOURCE = os.getenv("ANSWER_MODEL_SOURCE", "cloud").strip().lower()

In [ ]:
# Force local Ollama only (disable cloud path)
ANSWER_MODEL_SOURCE = "local"
OLLAMA_CLOUD_BASE_URL = ""
OLLAMA_CLOUD_API_KEY = ""
cloud_available = False
cloud_llm = None

# Re-bind models to local Ollama
local_llm = ChatOllama(
    model=LOCAL_LLM_MODEL,
    base_url=LOCAL_OLLAMA_BASE_URL,
    temperature=0
)

memory_llm = local_llm
answer_llm = local_llm
answer_llm_name = "local_ollama"

print("Cloud disabled.")
print("Memory LLM: local_ollama")
print("Answer LLM: local_ollama")

In [ ]:
# Section 4: Initialize local and cloud Ollama LLMs
local_llm = ChatOllama(
    model=LOCAL_LLM_MODEL,
    base_url=LOCAL_OLLAMA_BASE_URL,
    temperature=0
)

cloud_available = bool(OLLAMA_CLOUD_BASE_URL and OLLAMA_CLOUD_API_KEY)

if cloud_available:
    cloud_llm = ChatOllama(
        model=OLLAMA_CLOUD_MODEL,
        base_url=OLLAMA_CLOUD_BASE_URL,
        temperature=0,
        client_kwargs={
            "headers": {
                "Authorization": f"Bearer {OLLAMA_CLOUD_API_KEY}"
            }
        }
    )
else:
    cloud_llm = None

# Memo creation uses local LLM for fast local processing.
memory_llm = local_llm

# Final answer prefers cloud LLM, with local fallback.
if ANSWER_MODEL_SOURCE == "cloud" and cloud_llm is not None:
    answer_llm = cloud_llm
    answer_llm_name = "ollama_cloud"
else:
    answer_llm = local_llm
    answer_llm_name = "local_ollama"

print("Memory LLM: local_ollama")
print(f"Answer LLM: {answer_llm_name}")

In [ ]:
embeddings=OllamaEmbeddings(
    model=LOCAL_EMBED_MODEL,
    base_url=LOCAL_OLLAMA_BASE_URL
)

## Pure MemoRAG Workflow with StateGraph (PyPDFLoader + Multi-Query Loop)

This section is pure MemoRAG:
- build memory context from memory index
- generate retrieval cues from memory
- retrieve evidence using query + cues
- generate grounded answer
- self-refine retrieval if needed

It uses:
- `PyPDFLoader` for PDF input (default: `F:\\resume.pdf`)
- local Ollama for embeddings and memory formation
- local/cloud Ollama for answer generation (based on env config)
- infinite user query loop (`exit` to stop)

In [ ]:
# Section 13: Extra imports for pure MemoRAG graph workflow
from typing import TypedDict, List, Literal
from langgraph.graph import StateGraph, START, END
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# Section 14: Build MemoRAG indices from PDF with PyPDFLoader (parallel memory formation)
PDF_PATH = input("Enter PDF path (default F:\\resume.pdf): ").strip() or r"F:\resume.pdf"

loader = PyPDFLoader(PDF_PATH)
pdf_docs = loader.load()

pdf_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=1500
)
pdf_chunks = pdf_splitter.split_documents(pdf_docs)

# Memory formation from chunk windows
pdf_window_size = 3
pdf_windows = [pdf_chunks[i:i + pdf_window_size] for i in range(0, len(pdf_chunks), pdf_window_size)]

pdf_memory_prompt = ChatPromptTemplate.from_template(
    """
You are building compact global memory for long-context QA.
Summarize this passage into:
1) key facts
2) entities and relationships
3) retrieval cues for future questions

Passage:
{passage}
""".strip()
)

# Prepare all prompt messages
memory_msgs = [
    pdf_memory_prompt.format_messages(
        passage="\n\n".join(x.page_content for x in win)
    )
    for win in pdf_windows
]

# Parallel generation (with safe fallback)
max_concurrency = min(12, max(1, len(memory_msgs)))
try:
    memory_outputs = memory_llm.batch(
        memory_msgs,
        config={"max_concurrency": max_concurrency}
    )
except Exception:
    memory_outputs = [memory_llm.invoke(msg) for msg in memory_msgs]

pdf_memory_docs = [
    Document(
        page_content=out.content,
        metadata={"memory_id": idx, "source": "pdf_memory"}
    )
    for idx, out in enumerate(memory_outputs)
]

pdf_memory_store = FAISS.from_documents(pdf_memory_docs, embeddings)
pdf_evidence_store = FAISS.from_documents(pdf_chunks, embeddings)

pdf_memory_retriever = pdf_memory_store.as_retriever(search_kwargs={"k": 4})
pdf_evidence_retriever = pdf_evidence_store.as_retriever(search_kwargs={"k": 6})

print(f"Loaded pages: {len(pdf_docs)}")
print(f"Evidence chunks: {len(pdf_chunks)}")
print(f"Memory units: {len(pdf_memory_docs)}")
print(f"Memory build max_concurrency: {max_concurrency}")

In [ ]:
# Section 15: State + minimal functions (Pure MemoRAG workflow)
class MemoRAGState(TypedDict, total=False):
    query: str
    memory_context: str
    retrieval_cues: str
    docs: List[Document]
    answer: str
    route_decision: Literal["refine", "__end__"]
    attempts: int
    max_attempts: int


def build_memory(state: MemoRAGState) -> MemoRAGState:
    query = state["query"]
    mem_docs = pdf_memory_retriever.invoke(query)
    memory_context = "\n\n".join(d.page_content for d in mem_docs)
    return {
        "memory_context": memory_context,
        "attempts": state.get("attempts", 0),
        "max_attempts": state.get("max_attempts", 2),
    }


def generate_clues(state: MemoRAGState) -> MemoRAGState:
    query = state["query"]
    clue_prompt = ChatPromptTemplate.from_template(
        """
You are MemoRAG clue generator.
Using the query and memory context, generate 4 to 8 concise retrieval clues.
Return bullet points only.

Query:
{query}

Memory Context:
{memory_context}
""".strip()
    )
    msg = clue_prompt.format_messages(query=query, memory_context=state.get("memory_context", ""))
    cues = answer_llm.invoke(msg).content.strip()
    return {"retrieval_cues": cues}


def retrieve_docs(state: MemoRAGState) -> MemoRAGState:
    query = state["query"]
    cues = state.get("retrieval_cues", "")
    retrieval_query = f"Question: {query}\n\nClues:\n{cues}"
    docs = pdf_evidence_retriever.invoke(retrieval_query)
    return {"docs": docs}


def generate_answer(state: MemoRAGState) -> MemoRAGState:
    query = state["query"]
    evidence = "\n\n".join(d.page_content for d in state.get("docs", []))
    prompt = ChatPromptTemplate.from_template(
        """
Answer the query using only the evidence.
If evidence is insufficient, explicitly write: INSUFFICIENT_EVIDENCE.

Query:
{query}

Memory Context:
{memory_context}

Evidence:
{evidence}
""".strip()
    )
    msg = prompt.format_messages(
        query=query,
        memory_context=state.get("memory_context", ""),
        evidence=evidence,
    )
    ans = answer_llm.invoke(msg).content
    return {"answer": ans}


def refine_clues(state: MemoRAGState) -> MemoRAGState:
    refine_prompt = ChatPromptTemplate.from_template(
        """
Refine retrieval clues to improve evidence quality.
Use query, memory context, previous clues, and answer.
Return 4 to 8 improved bullet clues only.

Query:
{query}

Memory Context:
{memory_context}

Previous Clues:
{clues}

Previous Answer:
{answer}
""".strip()
    )
    msg = refine_prompt.format_messages(
        query=state["query"],
        memory_context=state.get("memory_context", ""),
        clues=state.get("retrieval_cues", ""),
        answer=state.get("answer", ""),
    )
    refined = answer_llm.invoke(msg).content.strip()
    return {
        "retrieval_cues": refined,
        "attempts": state.get("attempts", 0) + 1,
    }


def route_after_answer(state: MemoRAGState) -> str:
    attempts = state.get("attempts", 0)
    max_attempts = state.get("max_attempts", 2)
    answer = state.get("answer", "")

    if attempts >= max_attempts:
        return "__end__"

    if "INSUFFICIENT_EVIDENCE" in answer:
        return "refine"

    judge_prompt = ChatPromptTemplate.from_template(
        """
Decide whether retrieval should be refined.
Return exactly one token:
- refine
- __end__

Query:
{query}

Answer:
{answer}
""".strip()
    )
    msg = judge_prompt.format_messages(query=state["query"], answer=answer)
    raw = answer_llm.invoke(msg).content.strip().lower()
    return "refine" if "refine" in raw else "__end__"

In [ ]:
# =========================
# MemoRAG Graph
# =========================

builder = StateGraph(MemoRAGState)

# Core nodes
builder.add_node("build_memory", build_memory)          # global memory
builder.add_node("generate_clues", generate_clues)      # draft answers
builder.add_node("retrieve_docs", retrieve_docs)        # use clues
builder.add_node("generate_answer", generate_answer)    # final answer

# Optional refinement loop (like RLGF-inspired)
builder.add_node("refine_clues", refine_clues)


# =========================
# FLOW
# =========================

builder.add_edge(START, "build_memory")
builder.add_edge("build_memory", "generate_clues")
builder.add_edge("generate_clues", "retrieve_docs")
builder.add_edge("retrieve_docs", "generate_answer")

# Optional loop (advanced MemoRAG)
builder.add_conditional_edges(
    "generate_answer",
    route_after_answer,
    {
        "refine": "refine_clues",
        "__end__": END,
    },
)

builder.add_edge("refine_clues", "retrieve_docs")

app = builder.compile()
app

In [ ]:
# Section 17: Infinite user query loop (type exit to stop)
while True:
    user_query = input("\nAsk your question (or type 'exit'): ").strip()
    if user_query.lower() in {"exit", "quit", "q"}:
        print("Stopped.")
        break
    if not user_query:
        print("Please enter a valid query.")
        continue

    state_in = {
        "query": user_query,
        "attempts": 0,
        "max_attempts": 2,
    }
    state_out = app.invoke(state_in)

    print("\n=== Final Answer ===")
    print(state_out.get("answer", "No answer generated."))

    if state_out.get("retrieval_cues"):
        print("\n=== Retrieval Cues ===")
        print(state_out["retrieval_cues"])